IMPORT LIBRARIES

In [1]:
"""
Retail Demand Forecasting
Data Loading and Preprocessing Module
"""

# ==========================================
# Data Manipulation
# ==========================================

import pandas as pd
import numpy as np

# ==========================================
# Visualization
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Statistical Analysis
# ==========================================

from scipy import stats

# ==========================================
# Date & Time Handling
# ==========================================

from datetime import datetime

# ==========================================
# Seasonal Decomposition
# ==========================================

from statsmodels.tsa.seasonal import seasonal_decompose

# ==========================================
# Machine Learning Utilities
# ==========================================

from sklearn.model_selection import train_test_split

# ==========================================
# Logging
# ==========================================

import logging

# ==========================================
# System Utilities
# ==========================================

import os
import warnings

# ==========================================
# Warning Configuration
# ==========================================

warnings.filterwarnings("ignore")

# ==========================================
# Plot Configuration
# ==========================================

plt.style.use("ggplot")

# ==========================================
# Logging Configuration
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

# ==========================================
# Random Seed
# ==========================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logger.info("Libraries Imported Successfully")

2026-06-10 20:16:30,628 - INFO - Libraries Imported Successfully


Dataset Loading

In [2]:
import os

DATA_PATH = "../data/raw/Walmart.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("Dataset loaded successfully!")
else:
    print(f"Dataset not found at: {DATA_PATH}")

Dataset loaded successfully!


BASIC DATA INSPECTION

In [3]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(6435, 8)


In [4]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [5]:
df.tail()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
6430,45,28-09-2012,713173.95,0,64.88,3.997,192.013558,8.684
6431,45,05-10-2012,733455.07,0,64.89,3.985,192.170412,8.667
6432,45,12-10-2012,734464.36,0,54.47,4.000,192.327265,8.667
6433,45,19-10-2012,718125.53,0,56.47,3.969,192.330854,8.667
6434,45,26-10-2012,760281.43,0,58.85,3.882,192.308899,8.667


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Store,6435.0,2.300000e+01,12.988182,1.000,12.000,23.000000,3.400000e+01,4.500000e+01
Weekly_Sales,6435.0,1.046965e+06,564366.622054,209986.250,553350.105,960746.040000,1.420159e+06,3.818686e+06
Holiday_Flag,6435.0,6.993007e-02,0.255049,0.000,0.000,0.000000,0.000000e+00,1.000000e+00
Temperature,6435.0,6.066378e+01,18.444933,-2.060,47.460,62.670000,7.494000e+01,1.001400e+02
Fuel_Price,6435.0,3.358607e+00,0.459020,2.472,2.933,3.445000,3.735000e+00,4.468000e+00
CPI,6435.0,1.715784e+02,39.356712,126.064,131.735,182.616521,2.127433e+02,2.272328e+02
Unemployment,6435.0,7.999151e+00,1.875885,3.879,6.891,7.874000,8.622000e+00,1.431300e+01


CHECK MISSING VALUES

In [8]:
missing_values = df.isnull().sum()

missing_report = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": round(
        (missing_values / len(df)) * 100,
        2
    )
})

missing_report

,Missing Count,Missing Percentage
Store,0,0.0
Date,0,0.0
Weekly_Sales,0,0.0
Holiday_Flag,0,0.0
Temperature,0,0.0
Fuel_Price,0,0.0
CPI,0,0.0
Unemployment,0,0.0


CHECK DUPLICATES

In [10]:
duplicate_count = df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

Duplicate Rows: 0


CONVERT DATE COLUMN

In [12]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y"
)

print(df["Date"].dtype)

datetime64[ns]


CREATE TIME FEATURES

In [13]:
# ==========================================
# Time-Based Features
# ==========================================

df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

df["Quarter"] = df["Date"].dt.quarter

df["Day"] = df["Date"].dt.day

df["DayOfWeek"] = df["Date"].dt.dayofweek

df["WeekendFlag"] = (
    df["DayOfWeek"] >= 5
).astype(int)

print("Time Features Created Successfully!")

Time Features Created Successfully!


In [14]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Week,Quarter,Day,DayOfWeek,WeekendFlag
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,5,4,0
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,6,1,12,4,0
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,7,1,19,4,0
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,8,1,26,4,0
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,9,1,5,4,0
